# Silver → Gold

Modelo dimensional, marts para o Power BI e tabelas de features para o ML.

| Bloco | Tabelas |
|---|---|
| Dimensões | `dim_data`, `dim_material`, `dim_local`, `dim_centro_custo`, `dim_fornecedor`, `dim_operacao` |
| Fatos | `fact_movimento`, `fact_compra`, `fact_estoque_diario`, `fact_lote_posicao` |
| Qualidade | `qualidade_conciliacao_saldo` |
| Marts | `mart_consumo_mensal`, `mart_indicadores_estoque`, `mart_curva_abc_xyz`, `mart_fornecedor` |
| ML | `ml_demanda_semanal`, `ml_perfil_item` |

Chaves naturais do ERP (`cd_material`, `cd_local_estoque` etc.), sem chave substituta.

## 1. Parâmetros e período

In [0]:
from pyspark.sql import functions as F, Window

dbutils.widgets.text("catalogo", "estoque_hospitalar", "Catálogo")
CATALOGO = dbutils.widgets.get("catalogo").strip()
spark.sql(f"USE CATALOG {CATALOGO}")


def gravar(df, tabela, comentario):
    (
        df.withColumn("_dt_processamento", F.current_timestamp())
        .write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"gold.{tabela}")
    )
    spark.sql(f"COMMENT ON TABLE gold.{tabela} IS '{comentario}'")
    print(f"gold.{tabela:<30} {spark.table(f'gold.{tabela}').count():>12,}")


limites = spark.table("silver.tb_movimento_estoque").agg(
    F.to_date(F.min("dt_movimento")).alias("ini"), F.to_date(F.max("dt_movimento")).alias("fim")
).first()
DT_INICIO, DT_FIM = limites["ini"], limites["fim"]
print(f"Período: {DT_INICIO} a {DT_FIM}")

Período: 2024-10-01 a 2026-09-30


## 2. Dimensões

In [0]:
FERIADOS = [
    "2024-01-01", "2024-02-12", "2024-02-13", "2024-03-29", "2024-04-21", "2024-05-01", "2024-05-30",
    "2024-09-07", "2024-10-12", "2024-11-02", "2024-11-15", "2024-11-20", "2024-12-25",
    "2025-01-01", "2025-03-03", "2025-03-04", "2025-04-18", "2025-04-21", "2025-05-01", "2025-06-19",
    "2025-09-07", "2025-10-12", "2025-11-02", "2025-11-15", "2025-11-20", "2025-12-25",
    "2026-01-01", "2026-02-16", "2026-02-17", "2026-04-03", "2026-04-21", "2026-05-01", "2026-06-04",
    "2026-09-07", "2026-10-12", "2026-11-02", "2026-11-15", "2026-11-20", "2026-12-25",
]
DIAS_SEMANA = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]

# dayofweek do Spark começa no domingo (1); convertido para segunda = 1
dia_iso = (F.dayofweek("dt_data") + 5) % 7 + 1

# Começa um mês antes do período para incluir a data do saldo de abertura
dim_data = (
    spark.sql(f"SELECT explode(sequence(date'{DT_INICIO}' - INTERVAL 1 MONTH, date'{DT_FIM}', INTERVAL 1 DAY)) AS dt_data")
    .withColumn("nr_ano", F.year("dt_data"))
    .withColumn("nr_mes", F.month("dt_data"))
    .withColumn("ds_ano_mes", F.date_format("dt_data", "yyyy-MM"))
    .withColumn("dt_inicio_mes", F.trunc("dt_data", "month"))
    .withColumn("nr_semana_ano", F.weekofyear("dt_data"))
    .withColumn("dt_inicio_semana", F.date_trunc("week", "dt_data").cast("date"))
    .withColumn("nr_dia_semana", dia_iso)
    .withColumn("ds_dia_semana", F.element_at(F.array(*[F.lit(d) for d in DIAS_SEMANA]), dia_iso))
    .withColumn("fl_fim_semana", dia_iso >= 6)
    .withColumn("fl_feriado", F.col("dt_data").cast("string").isin(FERIADOS))
    .withColumn("fl_dia_util", ~F.col("fl_fim_semana") & ~F.col("fl_feriado"))
)
gravar(dim_data, "dim_data", "Calendário com semana, mês, feriados nacionais e dias úteis")

gold.dim_data                                760


In [0]:
estabelecimento = spark.table("silver.tb_estabelecimento").select("cd_estabelecimento", "ds_estabelecimento")

dim_material = spark.table("silver.tb_material").drop("_arquivo_origem", "_dt_ingestao", "_dt_processamento")
gravar(dim_material, "dim_material", "Materiais com hierarquia, unidades e situação")

dim_local = (
    spark.table("silver.tb_local_estoque")
    .join(estabelecimento, "cd_estabelecimento", "left")
    .select("cd_local_estoque", "ds_local_estoque", "ds_tipo_local", "cd_estabelecimento", "ds_estabelecimento", "fl_ativo")
)
gravar(dim_local, "dim_local", "Locais de estoque com estabelecimento")

dim_centro_custo = (
    spark.table("silver.tb_centro_custo")
    .join(estabelecimento, "cd_estabelecimento", "left")
    .select("cd_centro_custo", "ds_centro_custo", "cd_estabelecimento", "ds_estabelecimento")
)
gravar(dim_centro_custo, "dim_centro_custo", "Centros de custo com estabelecimento")

dim_fornecedor = spark.table("silver.tb_fornecedor").drop("_arquivo_origem", "_dt_ingestao", "_dt_processamento")
gravar(dim_fornecedor, "dim_fornecedor", "Fornecedores")

# Agrupa as 13 operações do ERP em categorias de negócio
CATEGORIAS = {
    1: "Consumo setor", 4: "Consumo paciente", 5: "Devolução", 17: "Devolução",
    87: "Perda", 91: "Perda", 121: "Perda", 137: "Perda",
    92: "Vencido", 93: "Vencido", 132: "Saída administrativa", 141: "Saída administrativa",
    136: "Ajuste de entrada",
}
mapa_categoria = F.create_map(*[F.lit(x) for kv in CATEGORIAS.items() for x in kv])
dim_operacao = (
    spark.table("silver.tb_operacao_estoque")
    .select("cd_operacao", "ds_operacao", "ds_tipo_operacao", "fl_entrada", "nr_sinal_saldo")
    .withColumn("ds_categoria", F.coalesce(mapa_categoria[F.col("cd_operacao")], F.lit("Outros")))
    .withColumn("fl_consumo", F.col("cd_operacao").isin(1, 4, 5, 17))
    .withColumn("fl_perda", F.col("ds_categoria").isin("Perda", "Vencido"))
)
gravar(dim_operacao, "dim_operacao", "Operações com sinal no saldo e categoria de negócio")

gold.dim_material                            871
gold.dim_local                                18
gold.dim_centro_custo                         49
gold.dim_fornecedor                           47
gold.dim_operacao                             13


## 3. Fatos de movimento e compra

| Tabela | Grão |
|---|---|
| `fact_movimento` | Uma movimentação |
| `fact_compra` | Um item de nota fiscal |

In [0]:
operacao = spark.table("gold.dim_operacao").select("cd_operacao", "nr_sinal_saldo", "ds_categoria", "fl_consumo", "fl_perda")

# Consumo líquido: saídas para paciente e setor somam, devoluções subtraem
sinal_consumo = F.when(F.col("cd_operacao").isin(1, 4), 1).when(F.col("cd_operacao").isin(5, 17), -1).otherwise(0)

fact_movimento = (
    spark.table("silver.tb_movimento_estoque")
    .join(F.broadcast(operacao), "cd_operacao")
    .select(
        "nr_movimento_estoque",
        F.to_date("dt_movimento").alias("dt_data"),
        "dt_movimento",
        "dt_competencia",
        "cd_material",
        "cd_local_estoque",
        "cd_centro_custo",
        "cd_operacao",
        "ds_categoria",
        "fl_estorno",
        "nr_atendimento",
        "nr_seq_lote",
        "cd_conta_contabil",
        "qt_estoque",
        (F.col("qt_estoque") * F.col("nr_sinal_saldo")).alias("qt_efeito_saldo"),
        (F.col("qt_estoque") * sinal_consumo).alias("qt_consumo_liquido"),
        F.when(F.col("fl_perda"), F.col("qt_estoque")).otherwise(0.0).alias("qt_perda"),
        "vl_movimento",
        (F.col("vl_movimento") * F.col("nr_sinal_saldo")).alias("vl_efeito_saldo"),
        F.col("vl_consumo").alias("vl_consumo_liquido"),
        F.when(F.col("fl_perda"), F.col("vl_movimento")).otherwise(0.0).alias("vl_perda"),
        "fl_competencia_seguinte",
        "fl_data_movimento_corrigida",
    )
)
gravar(fact_movimento, "fact_movimento", "Movimentações com efeito no saldo, consumo líquido e perdas")

gold.fact_movimento                    2,656,470


In [0]:
nota = spark.table("silver.tb_nota_fiscal").select(
    "nr_seq_nota_fiscal", "nr_nota_fiscal", "cd_estabelecimento", "cd_fornecedor",
    "dt_emissao", "dt_entrada", "vl_mercadoria", "vl_frete",
)
fact_compra = (
    spark.table("silver.tb_nota_fiscal_item")
    .join(nota, "nr_seq_nota_fiscal")
    .select(
        "nr_seq_nota_fiscal",
        "nr_item_nf",
        "nr_nota_fiscal",
        F.to_date("dt_entrada").alias("dt_data"),
        "dt_emissao",
        "dt_entrada",
        F.datediff("dt_entrada", "dt_emissao").alias("qt_dias_emissao_entrada"),
        "cd_estabelecimento",
        "cd_fornecedor",
        "cd_material",
        "cd_local_estoque",
        "nr_seq_lote",
        "cd_unidade_compra",
        "qt_item_nf",
        "qt_item_estoque",
        "vl_unitario_compra",
        "vl_unitario_estoque",
        "vl_total_item",
        # Frete da nota rateado pela participação do item no valor das mercadorias
        F.round(F.col("vl_frete") * F.col("vl_total_item") / F.col("vl_mercadoria"), 2).alias("vl_frete_rateado"),
    )
)
gravar(fact_compra, "fact_compra", "Itens de compra com fornecedor, datas e custo unitário em unidade de estoque")

gold.fact_compra                          24,151


## 4. fact_estoque_diario

Saldo por item, local e dia: saldo de abertura + entradas por nota + movimentações com o sinal da operação. Valorizado pelo último custo médio conhecido.

| Indicador | Regra |
|---|---|
| Ruptura | Saldo zerado, consumo nos últimos 30 dias e saldo positivo em pelo menos metade dos 90 dias anteriores |
| Compra sob demanda | Saldo positivo em menos da metade dos 90 dias anteriores; não conta como ruptura |
| Item parado | Saldo positivo sem saída para consumo há mais de 90 dias |

In [0]:
saldo_silver = spark.table("silver.tb_saldo_estoque")
dt_abertura = saldo_silver.agg(F.min("dt_posicao")).first()[0]

abertura = (
    saldo_silver.filter(F.col("dt_posicao") == dt_abertura)
    .groupBy("cd_material", "cd_local_estoque")
    .agg(F.sum("qt_estoque").alias("qt_abertura"), F.sum("vl_estoque").cast("double").alias("vl_abertura"))
    .withColumn("vl_custo_abertura", F.col("vl_abertura") / F.col("qt_abertura"))
)

mov_dia = (
    spark.table("gold.fact_movimento")
    .groupBy("cd_material", "cd_local_estoque", "dt_data")
    .agg(
        F.sum("qt_efeito_saldo").alias("qt_movimento_liquido"),
        F.sum("qt_consumo_liquido").alias("qt_consumo_liquido"),
        F.sum("vl_consumo_liquido").alias("vl_consumo_liquido"),
        F.sum("qt_perda").alias("qt_perda"),
        F.sum("vl_perda").alias("vl_perda"),
        F.sum(F.when(F.col("qt_consumo_liquido") > 0, 1).otherwise(0)).alias("nr_saidas_consumo"),
        (F.sum(F.abs("vl_movimento")) / F.sum(F.abs("qt_estoque"))).alias("vl_custo_dia"),
    )
)

nf_dia = (
    spark.table("gold.fact_compra")
    .groupBy("cd_material", "cd_local_estoque", "dt_data")
    .agg(
        F.sum("qt_item_estoque").alias("qt_entrada_nf"),
        F.sum("vl_total_item").cast("double").alias("vl_entrada_nf"),
    )
)

# Todo par item/local com saldo, movimento ou compra recebe uma linha por dia do período
pares = (
    abertura.select("cd_material", "cd_local_estoque")
    .union(mov_dia.select("cd_material", "cd_local_estoque"))
    .union(nf_dia.select("cd_material", "cd_local_estoque"))
    .distinct()
)
calendario = spark.table("gold.dim_data").filter(F.col("dt_data").between(DT_INICIO, DT_FIM)).select("dt_data")

base = (
    pares.crossJoin(calendario)
    .join(mov_dia, ["cd_material", "cd_local_estoque", "dt_data"], "left")
    .join(nf_dia, ["cd_material", "cd_local_estoque", "dt_data"], "left")
    .join(abertura, ["cd_material", "cd_local_estoque"], "left")
    .fillna(0.0, subset=["qt_movimento_liquido", "qt_consumo_liquido", "vl_consumo_liquido", "qt_perda",
                         "vl_perda", "qt_entrada_nf", "vl_entrada_nf", "qt_abertura"])
    .fillna(0, subset=["nr_saidas_consumo"])
)

In [0]:
par = Window.partitionBy("cd_material", "cd_local_estoque").orderBy("dt_data")
acumulado = par.rowsBetween(Window.unboundedPreceding, Window.currentRow)
ultimos_30 = par.rowsBetween(-29, Window.currentRow)
anteriores_90 = par.rowsBetween(-90, -1)

custo_nf = F.when(F.col("qt_entrada_nf") > 0, F.col("vl_entrada_nf") / F.col("qt_entrada_nf"))

fact_estoque_diario = (
    base
    .withColumn("qt_saldo", F.round(F.col("qt_abertura") + F.sum(F.col("qt_entrada_nf") + F.col("qt_movimento_liquido")).over(acumulado), 4))
    # Prioridade do custo: movimentações do dia, saldo de abertura, última nota recebida
    .withColumn("vl_custo_medio", F.coalesce(
        F.last("vl_custo_dia", ignorenulls=True).over(acumulado),
        F.col("vl_custo_abertura"),
        F.last(custo_nf, ignorenulls=True).over(acumulado),
    ))
    .withColumn("vl_saldo", F.round(F.greatest(F.col("qt_saldo"), F.lit(0.0)) * F.col("vl_custo_medio"), 2))
    .withColumn("qt_consumo_30d", F.sum("qt_consumo_liquido").over(ultimos_30))
    .withColumn("dt_ultima_saida", F.last(F.when(F.col("nr_saidas_consumo") > 0, F.col("dt_data")), ignorenulls=True).over(acumulado))
    # Sem saída no período, conta a partir da véspera do início
    .withColumn("qt_dias_sem_saida", F.datediff("dt_data", F.coalesce("dt_ultima_saida", F.date_sub(F.lit(DT_INICIO), 1))))
    .withColumn("fl_saldo_zerado", F.col("qt_saldo") <= 0)
    .withColumn("pc_dias_com_saldo_90d", F.avg((F.col("qt_saldo") > 0).cast("int")).over(anteriores_90))
    .withColumn("fl_compra_sob_demanda", F.coalesce(F.col("pc_dias_com_saldo_90d") < 0.5, F.lit(False)))
    .withColumn("fl_ruptura", F.col("fl_saldo_zerado") & (F.col("qt_consumo_30d") > 0) & ~F.col("fl_compra_sob_demanda"))
    .withColumn("fl_parado", (F.col("qt_saldo") > 0) & (F.col("qt_dias_sem_saida") > 90))
    .select(
        "dt_data", "cd_material", "cd_local_estoque",
        "qt_entrada_nf", "vl_entrada_nf", "qt_movimento_liquido", "qt_consumo_liquido", "vl_consumo_liquido",
        "qt_perda", "vl_perda", "qt_saldo", "vl_custo_medio", "vl_saldo",
        "qt_consumo_30d", "dt_ultima_saida", "qt_dias_sem_saida", "pc_dias_com_saldo_90d",
        "fl_saldo_zerado", "fl_compra_sob_demanda", "fl_ruptura", "fl_parado",
    )
)
gravar(fact_estoque_diario, "fact_estoque_diario", "Saldo diário reconstruído por item e local, com ruptura e itens parados")

gold.fact_estoque_diario               1,847,630


## 5. Conciliação com o saldo do ERP

Saldo reconstruído no último dia de cada mês comparado ao fechamento mensal do ERP.

In [0]:
fim_mes = spark.table("gold.dim_data").groupBy("dt_inicio_mes").agg(F.max("dt_data").alias("dt_posicao"))

reconstruido = (
    spark.table("gold.fact_estoque_diario")
    .join(fim_mes, F.col("dt_data") == F.col("dt_posicao"))
    .select("dt_posicao", "cd_material", "cd_local_estoque", F.col("qt_saldo").alias("qt_reconstruido"))
)
erp = (
    saldo_silver.filter(F.col("dt_posicao") > dt_abertura)
    .groupBy("dt_posicao", "cd_material", "cd_local_estoque")
    .agg(F.round(F.sum("qt_estoque"), 4).alias("qt_saldo_erp"))
)
# Full join: um par pode existir só de um lado (saldo zerado não aparece no fechamento do ERP)
conciliacao = (
    reconstruido.join(erp, ["dt_posicao", "cd_material", "cd_local_estoque"], "full")
    .fillna(0.0, subset=["qt_reconstruido", "qt_saldo_erp"])
    .withColumn("qt_diferenca", F.round(F.col("qt_reconstruido") - F.col("qt_saldo_erp"), 4))
    .withColumn("fl_divergente", F.abs("qt_diferenca") > 0.001)
)
gravar(conciliacao, "qualidade_conciliacao_saldo", "Saldo reconstruído no fim de cada mês comparado ao saldo do ERP")

display(
    spark.table("gold.qualidade_conciliacao_saldo")
    .groupBy("dt_posicao")
    .agg(F.count("*").alias("pares"), F.sum(F.col("fl_divergente").cast("int")).alias("divergentes"),
         F.max(F.abs("qt_diferenca")).alias("maior_diferenca"))
    .orderBy("dt_posicao")
)

gold.qualidade_conciliacao_saldo          60,744


dt_posicao,pares,divergentes,maior_diferenca
2024-10-31,2531,0,0.0
2024-11-30,2531,0,0.0
2024-12-31,2531,0,0.0
2025-01-31,2531,0,0.0
2025-02-28,2531,0,0.0
2025-03-31,2531,0,0.0
2025-04-30,2531,0,0.0
2025-05-31,2531,0,0.0
2025-06-30,2531,0,0.0
2025-07-31,2531,0,0.0


## 6. fact_lote_posicao

Posição mensal por lote com dias para vencer na data do fechamento.

In [0]:
lote = spark.table("silver.tb_lote_fornecedor").select("nr_seq_lote", "ds_lote_fornecedor", "dt_fabricacao", "dt_validade", "cd_fornecedor")
dias_vencer = F.datediff("dt_validade", "dt_posicao")

fact_lote_posicao = (
    saldo_silver
    .join(lote, "nr_seq_lote", "left")
    .withColumn("qt_dias_para_vencer", dias_vencer)
    .withColumn("ds_faixa_vencimento",
                F.when(F.col("dt_validade").isNull(), "Sem validade")
                .when(dias_vencer <= 0, "Vencido")
                .when(dias_vencer <= 30, "Até 30 dias")
                .when(dias_vencer <= 90, "31 a 90 dias")
                .when(dias_vencer <= 180, "91 a 180 dias")
                .otherwise("Acima de 180 dias"))
    .withColumn("fl_vencido", F.coalesce(dias_vencer <= 0, F.lit(False)))
    .select(
        "dt_posicao", "dt_competencia", "cd_estabelecimento", "cd_local_estoque", "cd_material", "nr_seq_lote",
        "ds_lote_fornecedor", "cd_fornecedor", "dt_fabricacao", "dt_validade", "qt_dias_para_vencer",
        "ds_faixa_vencimento", "fl_vencido", "qt_estoque", "vl_custo_medio", "vl_estoque",
    )
)
gravar(fact_lote_posicao, "fact_lote_posicao", "Posição mensal por lote com prazo e faixa de vencimento")

gold.fact_lote_posicao                    95,519


## 7. Marts para o Power BI

| Indicador | Cálculo |
|---|---|
| Giro | Consumo do mês / estoque médio |
| Cobertura | Saldo final / consumo médio diário do mês |
| Curva ABC | Valor consumido em 12 meses: A até 80% acumulado, B até 95%, C o restante |
| Classe XYZ | Coeficiente de variação da demanda semanal: X até 0,5, Y até 1,0, Z acima |

In [0]:
mart_consumo_mensal = (
    spark.table("gold.fact_movimento")
    .filter(F.col("ds_categoria").isin("Consumo paciente", "Consumo setor", "Devolução"))
    .withColumn("dt_inicio_mes", F.trunc("dt_data", "month"))
    .groupBy("dt_inicio_mes", "cd_material", "cd_local_estoque", "cd_centro_custo")
    .agg(
        F.round(F.sum("qt_consumo_liquido"), 4).alias("qt_consumo_liquido"),
        F.round(F.sum("vl_consumo_liquido"), 2).alias("vl_consumo_liquido"),
        F.round(F.sum(F.when(F.col("ds_categoria") == "Devolução", F.col("qt_estoque")).otherwise(0.0)), 4).alias("qt_devolvida"),
        F.countDistinct("nr_atendimento").alias("nr_atendimentos"),
        F.count("*").alias("nr_movimentos"),
    )
)
gravar(mart_consumo_mensal, "mart_consumo_mensal", "Consumo líquido mensal por item, local e centro de custo")

gold.mart_consumo_mensal                 125,163


In [0]:
diario = spark.table("gold.fact_estoque_diario").withColumn("dt_inicio_mes", F.trunc("dt_data", "month"))

mart_indicadores_estoque = (
    diario.groupBy("dt_inicio_mes", "cd_material", "cd_local_estoque")
    .agg(
        F.count("*").alias("nr_dias"),
        F.max_by("qt_saldo", "dt_data").alias("qt_saldo_final"),
        F.max_by("vl_saldo", "dt_data").alias("vl_saldo_final"),
        F.avg(F.greatest("qt_saldo", F.lit(0.0))).alias("qt_estoque_medio"),
        F.avg("vl_saldo").alias("vl_estoque_medio"),
        F.sum("qt_entrada_nf").alias("qt_entrada_nf"),
        F.sum("vl_entrada_nf").alias("vl_entrada_nf"),
        F.sum("qt_consumo_liquido").alias("qt_consumo_liquido"),
        F.sum("vl_consumo_liquido").alias("vl_consumo_liquido"),
        F.sum("qt_perda").alias("qt_perda"),
        F.sum("vl_perda").alias("vl_perda"),
        F.sum(F.col("fl_ruptura").cast("int")).alias("nr_dias_ruptura"),
        F.max_by("qt_dias_sem_saida", "dt_data").alias("qt_dias_sem_saida"),
        F.max_by("fl_parado", "dt_data").alias("fl_parado"),
    )
    .withColumn("nr_giro", F.when(F.col("vl_estoque_medio") > 0, F.col("vl_consumo_liquido") / F.col("vl_estoque_medio")))
    .withColumn("qt_dias_cobertura", F.when(F.col("qt_consumo_liquido") > 0,
                F.col("qt_saldo_final") / (F.col("qt_consumo_liquido") / F.col("nr_dias"))))
    .withColumn("qt_dias_estoque", F.when(F.col("vl_consumo_liquido") > 0,
                F.col("vl_estoque_medio") / (F.col("vl_consumo_liquido") / F.col("nr_dias"))))
)
gravar(mart_indicadores_estoque, "mart_indicadores_estoque", "Indicadores mensais de estoque: giro, cobertura, ruptura e perdas")

gold.mart_indicadores_estoque             60,744


In [0]:
# Semanas de segunda a domingo; as das pontas do período ficam incompletas e são marcadas
semanas = (
    spark.table("gold.dim_data")
    .filter(F.col("dt_data").between(DT_INICIO, DT_FIM))
    .groupBy("dt_inicio_semana")
    .agg(
        F.count("*").alias("nr_dias_periodo"),
        F.sum(F.col("fl_feriado").cast("int")).alias("nr_feriados"),
        F.first("nr_semana_ano").alias("nr_semana_ano"),
        F.month(F.max("dt_data")).alias("nr_mes"),
    )
    .withColumn("fl_semana_completa", F.col("nr_dias_periodo") == 7)
)

itens_consumo = spark.table("gold.fact_movimento").filter("ds_categoria IN ('Consumo paciente', 'Consumo setor')").select("cd_material").distinct()

consumo_semana = (
    spark.table("gold.fact_estoque_diario")
    .withColumn("dt_inicio_semana", F.date_trunc("week", "dt_data").cast("date"))
    .groupBy("cd_material", "dt_inicio_semana")
    .agg(
        F.sum("qt_consumo_liquido").alias("qt_consumo"),
        F.sum("vl_consumo_liquido").alias("vl_consumo"),
        F.sum(F.col("fl_ruptura").cast("int")).alias("nr_dias_ruptura"),
        F.sum("vl_saldo").alias("_vl_saldo"),
        F.sum(F.greatest("qt_saldo", F.lit(0.0))).alias("_qt_saldo"),
    )
)

# Série completa: semanas sem consumo entram com zero
demanda_semanal = (
    itens_consumo.crossJoin(semanas)
    .join(consumo_semana, ["cd_material", "dt_inicio_semana"], "left")
    .fillna(0.0, subset=["qt_consumo", "vl_consumo", "_vl_saldo", "_qt_saldo"])
    .fillna(0, subset=["nr_dias_ruptura"])
    .withColumn("vl_custo_medio", F.when(F.col("_qt_saldo") > 0, F.col("_vl_saldo") / F.col("_qt_saldo")))
    .drop("_vl_saldo", "_qt_saldo")
)

In [0]:
inicio_12m = F.add_months(F.lit(DT_FIM), -12)

abc = (
    spark.table("gold.fact_movimento")
    .filter(F.col("dt_data") > inicio_12m)
    .groupBy("cd_material")
    .agg(F.sum("vl_consumo_liquido").alias("vl_consumo_12m"), F.sum("qt_consumo_liquido").alias("qt_consumo_12m"))
    .filter(F.col("vl_consumo_12m") > 0)
)
ordem_valor = Window.orderBy(F.col("vl_consumo_12m").desc()).rowsBetween(Window.unboundedPreceding, Window.currentRow)
abc = (
    abc.withColumn("pc_acumulado", F.sum("vl_consumo_12m").over(ordem_valor) / F.sum("vl_consumo_12m").over(Window.partitionBy()))
    .withColumn("ds_classe_abc", F.when(F.col("pc_acumulado") <= 0.80, "A").when(F.col("pc_acumulado") <= 0.95, "B").otherwise("C"))
)

xyz = (
    demanda_semanal
    .filter(F.col("fl_semana_completa") & (F.col("dt_inicio_semana") > inicio_12m))
    .groupBy("cd_material")
    .agg(F.avg("qt_consumo").alias("qt_media_semanal"), F.stddev("qt_consumo").alias("qt_desvio_semanal"))
    .withColumn("nr_cv_semanal", F.when(F.col("qt_media_semanal") > 0, F.col("qt_desvio_semanal") / F.col("qt_media_semanal")))
    .withColumn("ds_classe_xyz", F.when(F.col("nr_cv_semanal") <= 0.5, "X").when(F.col("nr_cv_semanal") <= 1.0, "Y").otherwise("Z"))
)

mart_curva_abc_xyz = (
    spark.table("gold.dim_material").select("cd_material", "ds_material", "ds_grupo")
    .join(abc, "cd_material", "inner")
    .join(xyz, "cd_material", "left")
    .withColumn("ds_classe_abc_xyz", F.concat("ds_classe_abc", F.coalesce("ds_classe_xyz", F.lit("Z"))))
)
gravar(mart_curva_abc_xyz, "mart_curva_abc_xyz", "Classificação ABC por valor consumido e XYZ por variabilidade da demanda, últimos 12 meses")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gold.mart_curva_abc_xyz                      604


In [0]:
ordem_compra = Window.partitionBy("cd_material").orderBy("dt_entrada", "nr_seq_nota_fiscal", "nr_item_nf")

mart_fornecedor = (
    spark.table("gold.fact_compra")
    # Variação de preço em relação à compra anterior do mesmo material
    .withColumn("vl_unitario_anterior", F.lag("vl_unitario_estoque").over(ordem_compra))
    .withColumn("pc_variacao_preco", F.col("vl_unitario_estoque") / F.col("vl_unitario_anterior") - 1)
    .withColumn("dt_inicio_mes", F.trunc("dt_data", "month"))
    .groupBy("dt_inicio_mes", "cd_fornecedor")
    .agg(
        F.sum("vl_total_item").alias("vl_comprado"),
        F.sum("vl_frete_rateado").alias("vl_frete"),
        F.countDistinct("nr_seq_nota_fiscal").alias("nr_notas"),
        F.count("*").alias("nr_itens"),
        F.countDistinct("cd_material").alias("nr_materiais"),
        F.avg("qt_dias_emissao_entrada").alias("qt_dias_medio_emissao_entrada"),
        F.avg("pc_variacao_preco").alias("pc_variacao_preco_media"),
    )
)
gravar(mart_fornecedor, "mart_fornecedor", "Compras mensais por fornecedor com prazo e variação de preço")

gold.mart_fornecedor                       1,099


## 8. Features para o ML

`ml_demanda_semanal`: item x semana, somando todos os locais. Lags e médias móveis usam apenas semanas anteriores. As classes ABC/XYZ cobrem os últimos 12 meses e servem para agrupar a avaliação, não como preditor.

In [0]:
serie = Window.partitionBy("cd_material").orderBy("dt_inicio_semana")

ml_demanda_semanal = demanda_semanal
for k in [1, 2, 4, 8, 12]:
    ml_demanda_semanal = ml_demanda_semanal.withColumn(f"qt_consumo_lag_{k}", F.lag("qt_consumo", k).over(serie))
ml_demanda_semanal = (
    ml_demanda_semanal
    # Janela termina em -1 para não usar a própria semana na média
    .withColumn("qt_media_movel_4", F.avg("qt_consumo").over(serie.rowsBetween(-4, -1)))
    .withColumn("qt_media_movel_12", F.avg("qt_consumo").over(serie.rowsBetween(-12, -1)))
    .withColumn("qt_desvio_movel_12", F.stddev("qt_consumo").over(serie.rowsBetween(-12, -1)))
    .withColumn("nr_dias_ruptura_lag_1", F.lag("nr_dias_ruptura", 1).over(serie))
    .join(spark.table("gold.dim_material").select("cd_material", "cd_grupo", "ds_grupo", "fl_ativo"), "cd_material", "left")
    .join(spark.table("gold.mart_curva_abc_xyz").select("cd_material", "ds_classe_abc", "ds_classe_xyz"), "cd_material", "left")
)
gravar(ml_demanda_semanal, "ml_demanda_semanal", "Demanda semanal por item com lags, médias móveis e calendário para forecast")

gold.ml_demanda_semanal                   64,050


In [0]:
perfil_semanal = (
    demanda_semanal.filter("fl_semana_completa")
    .groupBy("cd_material")
    .agg(
        F.sum("qt_consumo").alias("qt_consumo_total"),
        F.sum("vl_consumo").alias("vl_consumo_total"),
        F.avg("qt_consumo").alias("qt_media_semanal"),
        F.stddev("qt_consumo").alias("qt_desvio_semanal"),
        F.avg((F.col("qt_consumo") > 0).cast("int")).alias("pc_semanas_com_consumo"),
        F.sum("nr_dias_ruptura").alias("nr_dias_ruptura"),
        F.avg("vl_custo_medio").alias("vl_custo_medio"),
    )
    .withColumn("nr_cv_semanal", F.when(F.col("qt_media_semanal") > 0, F.col("qt_desvio_semanal") / F.col("qt_media_semanal")))
)
perfil_estoque = (
    spark.table("gold.mart_indicadores_estoque")
    .groupBy("cd_material")
    .agg(
        F.sum("vl_perda").alias("vl_perda_total"),
        F.avg("qt_dias_cobertura").alias("qt_dias_cobertura_medio"),
        F.avg("vl_estoque_medio").alias("vl_estoque_medio"),
        F.countDistinct("cd_local_estoque").alias("nr_locais"),
    )
)
ml_perfil_item = (
    perfil_semanal.join(perfil_estoque, "cd_material", "left")
    .withColumn("pc_perda", F.when(F.col("vl_consumo_total") > 0, F.col("vl_perda_total") / F.col("vl_consumo_total")))
    .join(spark.table("gold.dim_material").select("cd_material", "ds_material", "cd_grupo", "ds_grupo"), "cd_material", "left")
)
gravar(ml_perfil_item, "ml_perfil_item", "Perfil de consumo, custo e perdas por item para segmentação")

gold.ml_perfil_item                          610


## 9. Conferência

In [0]:
display(
    spark.table("gold.mart_indicadores_estoque")
    .groupBy("dt_inicio_mes")
    .agg(
        F.round(F.sum("vl_saldo_final"), 0).alias("vl_estoque_fim_mes"),
        F.round(F.sum("vl_consumo_liquido"), 0).alias("vl_consumo"),
        F.round(F.sum("vl_perda"), 0).alias("vl_perda"),
        F.sum("nr_dias_ruptura").alias("dias_ruptura"),
        F.sum(F.col("fl_parado").cast("int")).alias("itens_parados"),
    )
    .orderBy("dt_inicio_mes")
)

dt_inicio_mes,vl_estoque_fim_mes,vl_consumo,vl_perda,dias_ruptura,itens_parados
2024-10-01,886157.0,1010483.0,7043.0,639,0
2024-11-01,794478.0,1018884.0,3950.0,1044,0
2024-12-01,832008.0,950583.0,15417.0,886,476
2025-01-01,702127.0,910654.0,10178.0,735,473
2025-02-01,680663.0,1114857.0,7193.0,967,473
2025-03-01,839533.0,1092543.0,8649.0,1212,462
2025-04-01,904719.0,1264359.0,10340.0,906,458
2025-05-01,848148.0,1154563.0,8473.0,964,445
2025-06-01,948643.0,907641.0,4210.0,782,464
2025-07-01,850750.0,1028348.0,9084.0,859,441
